# 🦉 Strixlibe × Google Colab GPU LLM Server (Strata + Qwen3.8-Flash-Next 125B 超高知能版)
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab（**NVIDIA L4 24GB GPU + ハイメモリ推奨**）上で、最新の階層オフロード推論エンジン **Strata** を使用して 1250億パラメータの超巨大 MoE モデル **Qwen3.8-Flash-Next** をホストするノートブックです。

### ⚡ 爆速並列DL & 安定起動パイプライン（改善版）
- **aria2 16分割並列ダウンロード**: 68GB の巨大重みを単一スレッド（20分）ではなく、16マルチストリーム並列で取得（約3〜4分に爆速短縮）。
- **起動完了の確実なヘルスチェック**: 68GB の重みが GPU＋RAM に完全に展開され、`/v1/models` が応答するまで待機してから疎通テストを実行（HTTP 404 エラーを完全解消）。
- **1クリック完全自動完走**: 下のセルを 1 回実行するだけで最後までノンストップで自動完走します。

> **⚠️ ランタイム設定の注意**: Colab のメニュー「ランタイム」→「ランタイムのタイプを変更」で、**GPU（L4）** かつ **ハイメモリ（High-RAM: 約53GB）** を選択してください。

In [ ]:
#@title 🚀 1クリック完全自動起動（爆速並列DL ＆ 起動待機強化版） { run: "auto" }
import os
import sys
import time
import glob
import psutil
import subprocess
import requests
import re
from pathlib import Path

# ==================== 1. 設定項目 ====================
MODEL_FAMILY = "qwen" #@param ["qwen", "coder"]
QUANT_LEVEL = "IQ2_XS" #@param ["IQ2_XS", "Q2_0", "IQ3_XXS", "IQ3_S"]
CONTEXT_LENGTH = 32768 #@param [16384, 32768, 65536, 131072]
HF_TOKEN = "" #@param {type:"string"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 25 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]

print("="*65)
print("🦉 Strixlibe × Strata 125B 高速並列DL & 安定起動パイプライン")
print("="*65)

# ==================== 2. スペック確認 & 初期化 ====================
print("\n[*] [1/5] ハードウェア & メモリ確認中...")
sys.stdout.flush()
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"[+] Total System RAM: {total_ram_gb:.1f} GB")
if total_ram_gb < 30:
    print("⚠️ [WARNING] システムRAMが30GB未満です。「ランタイムのタイプを変更」で「ハイメモリ」を推奨します。")

print("[*] 残存プロセスとポート競合をクリーンアップ中...")
subprocess.run("pkill -9 -f strata || true", shell=True)
subprocess.run("pkill -9 -f cloudflared || true", shell=True)
subprocess.run("rm -f /content/strata.log", shell=True)

# ==================== 3. 依存パッケージ & 高速ダウンローダー導入 ====================
print("\n[*] [2/5] ツール・高速並列ダウンローダー (aria2) を準備中...")
sys.stdout.flush()
subprocess.run("apt-get update -qq && apt-get install -y -qq git zstd curl pciutils aria2 python3-venv", shell=True)
subprocess.run("pip install -q --no-warn-conflicts hf-transfer huggingface_hub", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# cloudflared 導入
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[*] cloudflared をダウンロード中...")
    subprocess.run("curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)

# Strata クローン
strata_dir = "/content/Strata"
if not os.path.exists(strata_dir):
    print("[*] Strata リポジトリをクローン中...")
    subprocess.run("git clone --depth 1 https://github.com/Niko1221/Strata.git /content/Strata", shell=True)
else:
    print("[*] 既存のStrataをクリーンリセット中...")
    subprocess.run("cd /content/Strata && git fetch origin && git reset --hard origin/main && git clean -fd", shell=True)

# ==================== 4. 爆速並列ダウンロード（21分 → 3〜4分） ====================
print(f"\n[*] [3/5] モデルの爆速並列ダウンロード開始: Qwen3.8-Flash-Next ({QUANT_LEVEL}) ...")
print("[*] aria2 の 16分割マルチストリームで数Gbps帯域をフル活用します...")
sys.stdout.flush()

models_dir = Path("/content/Strata/data/models")
models_dir.mkdir(parents=True, exist_ok=True)

hf_base = "https://huggingface.co/ISTA-DASLab/Qwen3.8-Flash-Next-GSQ-RCO-GGUF/resolve/ed59f92082b1e93c0e96d60a8b11aab089b52f09"
target_files = [
    f"Qwen3.8-Flash-Next-GSQ-RCO-{QUANT_LEVEL}-00001-of-00002.gguf",
    f"Qwen3.8-Flash-Next-GSQ-RCO-{QUANT_LEVEL}-00002-of-00002.gguf"
]

dl_urls = []
for tf in target_files:
    dst = models_dir / tf
    done_file = dst.with_name(dst.name + ".done")
    if dst.exists() and done_file.exists():
        print(f"[+] {tf} は既にダウンロード済みです。スキップします。")
    else:
        url = f"{hf_base}/{QUANT_LEVEL}/{tf}"
        dl_urls.append((url, tf))

if dl_urls:
    dl_list_file = "/content/aria2_downloads.txt"
    with open(dl_list_file, "w") as f:
        for url, tf in dl_urls:
            f.write(f"{url}\n  out={tf}\n  dir={models_dir}\n")
    header_opt = f"--header='Authorization: Bearer {HF_TOKEN}'" if HF_TOKEN else ""
    aria2_cmd = f"aria2c -i {dl_list_file} -x 16 -s 16 -j 2 -k 2M --summary-interval=10 {header_opt}"
    start_t = time.time()
    ret = subprocess.run(aria2_cmd, shell=True)
    elapsed = time.time() - start_t
    print(f"\n[+] 並列ダウンロード完了！所要時間: {elapsed/60:.1f} 分 (約 {elapsed:.0f} 秒)")
    for _, tf in dl_urls:
        dst = models_dir / tf
        dst.with_name(dst.name + ".done").write_text(time.strftime("%Y-%m-%d %H:%M"), encoding="utf-8")

print("\n[*] Strata エンジン設定 & MTPドラフトレイヤーを構成中...")
sys.stdout.flush()
setup_cmd = f"python3 setup.py --yes --family {MODEL_FAMILY} --model {QUANT_LEVEL} --context {CONTEXT_LENGTH} --vision no --host 0.0.0.0 --port 8080 --no-start"
subprocess.run(setup_cmd, shell=True, cwd=strata_dir)

# ==================== 5. Strata サーバー起動 & 確実な起動待機 ====================
print("\n[*] [4/5] Strata サーバーを起動中...")
sys.stdout.flush()

run_scripts = glob.glob(f"{strata_dir}/run-*.sh")
if run_scripts:
    launch_cmd = ["bash", run_scripts[0]]
    print(f"[*] 生成されたランチャーを使用: {run_scripts[0]}")
else:
    launch_cmd = ["python3", f"{strata_dir}/serve/server.py", "--engine", "strata", "--config", f"{strata_dir}/strata-qwen.json", "--port", "8080"]

strata_log = open("/content/strata.log", "w")
server_proc = subprocess.Popen(launch_cmd, cwd=strata_dir, stdout=strata_log, stderr=subprocess.STDOUT)

print("[*] 68GB の重みを GPU (24GB) と RAM (53GB) にマッピング中（約1〜2分）...")
print("[*] サーバーの完全リッスンを待機中...")
sys.stdout.flush()

ready = False
model_id = "qwen3.8-flash-next"
for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8080/v1/models", timeout=2)
        if r.status_code == 200:
            ready = True
            data = r.json()
            if "data" in data and len(data["data"]) > 0:
                model_id = data["data"][0].get("id", model_id)
            break
    except:
        pass
    if i > 0 and i % 10 == 0:
        print(f"[*] メモリ展開中... ({i*2} 秒経過)")
        sys.stdout.flush()
    time.sleep(2)

if not ready:
    print("[-] サーバー起動がタイムアウトしました。/content/strata.log の末尾を確認します:")
    subprocess.run("tail -n 30 /content/strata.log || true", shell=True)
else:
    print(f"[+] 🎉 Strata サーバーが正常起動しました！（モデル ID: {model_id}）")

print("\n[*] Cloudflare Tunnel を確立中...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:8080", "--http-host-header", "localhost:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if not tunnel_url:
    print("[-] トンネルURLの取得に失敗しました。ローカル接続のみ利用可能です。")
    api_base = "http://127.0.0.1:8080/v1"
else:
    api_base = f"{tunnel_url}/v1"
    print(f"\n🎉 ==========================================================")
    print(f"[+] Cloudflare Tunnel 確立成功！")
    print(f"[+] 公開 URL: {tunnel_url}")
    print(f"[+] OpenAI API Base: {api_base}")
    print(f"==========================================================\n")

# ==================== 6. 疎通推論テスト（125B ベンチマーク） ====================
print("[*] [5/5] 125B モデルへの疎通推論テストを実行中...")
sys.stdout.flush()

test_payload = {
    "model": model_id,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントの125B推論基盤テストです。自律ペンテストにおいてあなたの得意とする役割を日本語で簡潔に2文以内で述べてください。"}
    ],
    "max_tokens": 150,
    "temperature": 0.6
}

test_target = f"{api_base}/chat/completions" if tunnel_url else "http://127.0.0.1:8080/v1/chat/completions"

try:
    resp = requests.post(test_target, json=test_payload, timeout=120)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 🎉 125B 推論テスト大成功！モデルからの返答:")
        print("-" * 55)
        print(reply.strip())
        print("-" * 55)
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")

if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-strata",
            "base_url": api_base,
            "model_name": model_id,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print("[+] Strixlibe Dispatcher に登録完了！")
    except Exception as e:
        print(f"[-] Dispatcher 登録スキップ: {e}")

print("\n[OK] 全工程が完了しました！125B の知能が利用可能です。")


### 2. CU保護ウォッチドッグ（無通信時の自動停止）
※ 必要に応じてバックグラウンド監視を動かす場合は実行してください（放置時のCompute Unit消費を防止します）。

In [ ]:
import threading
import time
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold exceeded. Terminating runtime to preserve Compute Units.")
            runtime.unassign()
            break

t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] Watchdog is active.")
